In [ ]:
from transformers import pipeline

MODEL_DIR = "model"

fill_mask = pipeline(
    "fill-mask",
    model=MODEL_DIR,
    tokenizer=MODEL_DIR
)

top_k = 2

tests = [
    "Paris is the [MASK] of France.",
    "Transformers are very useful for [MASK] tasks.",
    "Artificial [MASK] is a branch of computer science focused on creating systems.",
    "Python is a popular programming [MASK].",
    "The sun rises in the [MASK].",
    "Water freezes at 0 [MASK].",
    "A dog is a common household [MASK].",
    "The Earth revolves around the [MASK].",
    "She drank a cup of hot [MASK].",
    "Machine learning models learn from [MASK].",
    "A teacher works at a [MASK].",
    "Birds can [MASK] in the sky.",
    "The opposite of hot is [MASK].",
    "Doctors work in a [MASK].",
]

for text in tests:
    print("INPUT:", text)
    preds = fill_mask(text, top_k=top_k)
    for item in preds:
        print(f"sequence: {item['sequence']} -> score: {item['score']:.4f}")
    print("=" * 60)

In [ ]:
from transformers import pipeline
clf = pipeline(
    "text-classification",
    model='classifier',
    tokenizer='classifier',
    device=0,
)

tests = [
    "I love this phone",
    "This service is terrible",
    "The movie was okay",
    "I really enjoyed the game",
    "The weather is nice today",
    "The meeting was long",
    "The product works well",
    "The app crashes a lot",

    "I love this product",
    "This phone is terrible",
    "The service was amazing",
    "The food was awful",
    "The movie was boring",
    "The update improved everything",
]

for i, output in enumerate(clf(tests)):
    print(f"""\
Text: {tests[i]}
Label: {output['label']} -> Score: {output['score']:.4f}
---""")

In [ ]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model="ner",
    tokenizer="ner",
    aggregation_strategy="simple",
)

samples = [
    "Barack Obama visited Paris and met Microsoft executives.",
    "Elon Musk is the CEO of Tesla.",
    "Apple released the new iPhone in California.",
    "Lionel Messi played football in Barcelona.",
    "Amazon opened a new office in London.",
    "Bill Gates founded Microsoft.",
    "Google is headquartered in Mountain View.",
    "Sundar Pichai works at Google.",
]

for text in samples:
    print(f"Text: {text}")
    ents = ner(text)

    if not ents:
        print("Entities: None")
    else:
        for e in ents:
            print(
                f"Entity: {e['word']} | "
                f"Type: {e['entity_group']} | "
                f"Score: {e['score']:.4f}"
            )

    print("-" * 60)

In [ ]:
from transformers import pipeline

qa = pipeline(
    "question-answering",
    model='qa',
    tokenizer='qa',
)

samples = [
    {
        "question": "What is the capital of France?",
        "context": "France is a country in Europe. Its capital is Paris.",
    },
    {
        "question": "Who created Python?",
        "context": "Python is a programming language created by Guido van Rossum.",
    },
    {
        "question": "Where is Google headquartered?",
        "context": "Google is a technology company headquartered in Mountain View, California.",
    },
    {
        "question": "What do transformers use?",
        "context": "Transformers use self-attention to model relationships between tokens.",
    },
    {
        "question": "Who wrote Hamlet?",
        "context": "William Shakespeare wrote Hamlet in the late 16th century.",
    },
    {
        "question": "What planet do humans live on?",
        "context": "Humans live on Earth, the third planet from the Sun.",
    },
    {
        "question": "Where does Messi play football?",
        "context": "Lionel Messi played football in Barcelona and Paris.",
    },
    {
        "question": "What freezes at 0 degrees Celsius?",
        "context": "Water freezes at 0 degrees Celsius under normal conditions.",
    },
        {
        "question": "What is the capital of Egypt?",
        "context": "Cairo is the capital and largest city of Egypt. Located in the north of the country along the Nile River, it is a major political, economic, and cultural hub, with a population exceeding 10 million people.",
    },
]

for item in samples:
    result = qa(question=item["question"], context=item["context"])
    print(f"Question: {item['question']}")
    print(f"Context: {item['context']}")
    print(f"Answer: {result['answer']} -> Score: {result['score']:.4f}")
    print("-" * 60)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

embed_model_id = "embed_model"

tokenizer = AutoTokenizer.from_pretrained(embed_model_id)
model = AutoModel.from_pretrained(embed_model_id).eval().cuda()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode(texts):
    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        out = model(**batch)
        emb = mean_pooling(out.last_hidden_state, batch["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1)

    return emb


pairs = [
    ("A Dog is an animal.", "a dog is an animal."),
    ("A dog is an animal.", "a dog is an animal."),
    ("A dog is an animal.", "a dog was an animal."),
    ("A dog is an animal.", "a cat is an animal."),
    ("A cow is an animal.", "a dog is an animal."),
    ("A cow is an animal.", "a cat is an animal."),
]

for a, b in pairs:
    emb = encode([a, b])
    sim = F.cosine_similarity(emb[0].unsqueeze(0), emb[1].unsqueeze(0)).item()

    print(f"Text 1: {a}")
    print(f"Text 2: {b}")
    print(f"Similarity: {sim:.4f}")
    print("-" * 60)